In [0]:
class Silver_drivers():
    main_path="/Volumes/formula1_race/default/formula1/"
    bronze_path = "formula1_race_project/bronze"
    silver_path = "formula1_race_project/silver"

    def __init__(self,folder):
         self.folder_name=folder
        
    def read_input(self):
        read_df=spark.read.table('formula1_race.bronze.drivers')
        return read_df
    
    def column_name_formating(self,read_df):
        colrename_df=read_df
        import re
        for c in colrename_df.columns:
            result = re.sub(r'([a-z])([A-Z])',r'\1,\2',c).lower().split(',')
            new_column="_".join(result)
            colrename_df=colrename_df.withColumnRenamed(c,new_column)
            colrename_df =(colrename_df.withColumnRenamed('name','driver_name')
                                      .withColumnRenamed('code','driver_code')
                                      .withColumnRenamed('number','driver_number')
                                     .withColumnRenamed('nationality','driver_nationality')
                          )
        return colrename_df
    
    def apply_transformations(self,colrename_df):
        from pyspark.sql.functions import round,col,when,concat,lit
        apply_tran_df=colrename_df.withColumn('driver_name',
                                              concat(col('driver_name.forename'),lit(" "),col('driver_name.surname')))
        apply_tran_df=apply_tran_df.withColumn('driver_code',when(col('driver_code').isin('\\N'),None).otherwise(col('driver_code')))
        apply_tran_df=apply_tran_df.select(col('driver_id'),col('driver_ref'),col('driver_number'),col('driver_code'),
                                           col('driver_name'),col('dob'),col('driver_nationality'),col('drivers_ingestion_date'),col('source'))
        
        return apply_tran_df
    
    def write_output(self,apply_tran_df):
        apply_tran_df.write.mode("overwrite").saveAsTable("formula1_race.silver.drivers")
        display(spark.sql("select count(*) from formula1_race.silver.drivers"))
        print("Data write into sliver drivers table is Done")
       

    
    def process(self):
        print("Started silver-ingestion-drivers  in ran....")
        read_df=self.read_input()
        colrename_df=self.column_name_formating(read_df)
        apply_tran_df=self.apply_transformations(colrename_df)
        self.write_output(apply_tran_df)
       
    

In [0]:
Silver_drivers_instance = Silver_drivers("drivers")
Silver_drivers_instance.process()